# EVASPA Evaporative Fraction model and configuration

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pprint import pprint

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
import xarray as xr
from matplotlib.colors import ListedColormap

In [ ]:
from sweat.evaspa.ef import (
    EFConfig,
    EFModel,
    get_variables_from_models,
    initialize,
    run,
)

### Generate test data 

In [ ]:
def setup_data(
    dry: tuple[float, float],
    wet: tuple[float, float],
    albedo: tuple[float, float],
    valid: tuple[float, float] = (0.0, 1.0),
    fcover: tuple[float, float] | None = None,
    size: int = 125,
    seed: int = 0,
) -> xr.Dataset:
    """
    Generate data for tests
    """

    def dry_edge(v: npt.ArrayLike) -> npt.NDArray:
        return dry[1] * v + dry[0]

    def wet_edge(v: npt.ArrayLike) -> npt.NDArray:
        return wet[1] * v + wet[0]

    np.random.seed(seed)
    lon = np.linspace(start=1.0, stop=2.0, num=size, dtype=np.float32)
    lat = np.linspace(start=43.0, stop=44.0, num=size, dtype=np.float32)
    valid_arr = np.random.choice(
        [0, 1], size=(size, size), p=[valid[0], valid[1]]
    ).astype(int)
    albedo_arr = np.random.uniform(
        low=albedo[0], high=albedo[1], size=(size, size)
    )
    lst_func = np.vectorize(
        lambda x: np.random.uniform(wet_edge(x), dry_edge(x))
    )
    lst_arr = lst_func(albedo_arr)
    if fcover is not None:
        fcover_arr = np.random.uniform(
            low=fcover[0], high=fcover[1], size=(size, size)
        )
        lst_func = np.vectorize(
            lambda x, y: np.random.uniform(
                max(wet_edge(x), wet_edge(y)), min(dry_edge(x), dry_edge(y))
            )
        )
        lst_arr = lst_func(fcover_arr, albedo_arr)
    ef_albedo_arr = (dry_edge(albedo_arr) - lst_arr) / (
        dry_edge(albedo_arr) - wet_edge(albedo_arr)
    )
    data_vars = {
        "albedo": (["lon", "lat"], albedo_arr),
        "lst": (["lon", "lat"], lst_arr),
        "valid": (["lon", "lat"], valid_arr),
        "flags": (["lon", "lat"], np.where(valid_arr, 0, 2).astype(int)),
        "ef_albedo": (["lon", "lat"], ef_albedo_arr),
    }
    if fcover is not None:
        ef_fcover_arr = (dry_edge(fcover_arr) - lst_arr) / (
            dry_edge(fcover_arr) - wet_edge(fcover_arr)
        )
        data_vars["fcover"] = (["lon", "lat"], fcover_arr)
        data_vars["ef_fcover"] = (["lon", "lat"], ef_fcover_arr)
    return xr.Dataset(
        data_vars=data_vars,
        coords={
            "lon": ("lon", lon),
            "lat": ("lat", lat),
        },
        attrs={"description": "Test data"},
    )

In [ ]:
# Generate data
data = setup_data(
    albedo=(0.0, 0.6),
    fcover=(0.0, 1.0),
    dry=(330.0, -10.0),
    wet=(300.0, 15.0),
    valid=(0.2, 0.8),
)

In [ ]:
data

In [ ]:
def plot(
    data: xr.Dataset,
    var_name: str | None = None,
    model: EFModel | None = None,
    save: bool = False,
):
    """
    Plot
    """
    # Figure dimensions
    fig = plt.figure(constrained_layout=True, figsize=(4, 3))

    ax = plt.gca()

    if var_name is None and model is None:
        msg = "Provide either var_name or model"
        raise ValueError(msg)
    lst = data["lst"].data

    if var_name is not None:
        var = data[var_name].data
        # plot all points
        ax.scatter(var, lst, s=20, alpha=1, color="moccasin", clip_on=False)

    # plot edges
    elif model is not None:
        var_name = model.var
        var = data[model.var].data
        # plot all points
        ax.scatter(var, lst, s=20, alpha=1, color="moccasin", clip_on=False)
        # edges coordinates
        var_max = np.round(np.ceil(np.nanmax(var) * 10) / 10, decimals=1)
        var_min = np.round(np.floor(np.nanmin(var) * 10) / 10, decimals=1)

        var_arr = np.linspace(var_min, var_max, num=20)
        tw_arr = model.twet(var_arr)
        td_arr = model.tdry(var_arr)

        ax.plot(
            var_arr,
            td_arr,
            color="tab:red",
            linewidth=3,
            label="dry edge",
            zorder=1,
        )
        ax.plot(
            var_arr,
            tw_arr,
            color="tab:blue",
            linewidth=3,
            label="wet edge",
            zorder=1,
        )

    # Labels
    ax.set_xlabel(var_name)
    ax.set_ylabel("Land Surface Temperature K")

    # title
    ax.set_title("Evaporative fraction method", fontsize=12)

    # savefig
    if save:
        destfile = "ef.png"
        fig.savefig(destfile)

    plt.show()


def plot_dataset(
    data: xr.Dataset,
):
    """
    Plot EF models
    """
    # Set figure subplots
    nb = len(data.data_vars)
    row = int(np.ceil(nb / 2))
    col = 2
    fig = plt.figure(figsize=(4 * col, 3 * row), constrained_layout=True)

    i = 0
    j = 0
    # Loop over variables
    for var in data.data_vars:
        ax = plt.subplot2grid((row, col), (i, j))
        if var == "valid":
            valid_cmap = ListedColormap(
                [(1.0, 0.0, 0.0, 1.0), (0.0, 1.0, 0.0, 1.0)]
            )
            data[var].plot(ax=ax, cmap=valid_cmap, add_colorbar=False)
        elif var == "flags":
            da = data[var]
            unique_combinations = np.unique(da.values)
            # Use a qualitative colormap with enough colors
            # (e.g., tab20 or a custom one)
            cmap = plt.get_cmap("tab20", len(unique_combinations))
            # Map each unique mask value to an integer index (for coloring)
            value_to_index = {v: i for i, v in enumerate(unique_combinations)}
            indexed_data = da.copy(
                data=np.vectorize(value_to_index.get)(da.values)
            )
            indexed_data.plot(ax=ax, cmap=cmap, add_colorbar=False)
            legend_patches = [
                mpatches.Patch(color=cmap(i), label=format(v, "08b"))
                for i, v in enumerate(unique_combinations)
            ]
            ax.legend(
                handles=legend_patches,
                title="Flags (bits)",
                bbox_to_anchor=(1, 1),
                loc="upper right",
                fontsize="small",
                fancybox=True,
                framealpha=0.5,
            )
        else:
            data[var].plot(ax=ax)
        j += 1
        if j == 2:
            j = 0
            i += 1
    # title
    fig.suptitle("Dataset", fontsize=12)

    plt.show()

In [ ]:
plot_dataset(data)

In [ ]:
plot(data, "albedo")

### Define configuration

In [ ]:
config = {
    "models": [
        {
            "name": "model1",
            "dry_edge": {
                "type": "LinearEdge",
                "config": {
                    "interval_type": "size",
                    "interval_nb": 20,
                    "percentile": 2,
                    "selection": "median",
                },
            },
            "wet_edge": {
                "type": "LinearEdge",
                "config": {
                    "interval_type": "size",
                    "interval_nb": 20,
                    "percentile": 2,
                    "selection": "median",
                },
            },
            "var": "fcover",
        },
        {
            "name": "model2",
            "dry_edge": {
                "type": "LinearEdge",
                "config": {
                    "interval_type": "density",
                    "interval_nb": 20,
                    "percentile": 5,
                    "selection": "median",
                },
            },
            "wet_edge": {
                "type": "FlatEdge",
            },
            "var": "albedo",
        },
    ],
    "options": {
        "filtering": {"lst": {"op": ">", "value": 310}},
        "selection": False,
        "merging": {
            "merging_method": "mean",
            "uncertainty_method": "interquartile",
        },
    },
}

In [ ]:
pprint(config, sort_dicts=False)

### Check configuration

In [ ]:
# Validate parameters config
ef_config = EFConfig.model_validate(config).model_dump()

In [ ]:
pprint(ef_config, sort_dicts=False)

### Run

Initialize

In [ ]:
models, options = initialize(config)

In [ ]:
get_variables_from_models(models=models)

In [ ]:
models

In [ ]:
options

Compute

In [ ]:
ef, ef_inst = run(models, data, **options)

In [ ]:
ef

In [ ]:
ef_inst

Visualize results

In [ ]:
plot_dataset(ef)

In [ ]:
plot(data, model=EFModel.create(ef["model1"].attrs))

In [ ]:
plot(data, model=EFModel.create(ef["model2"].attrs))

In [ ]:
plot_dataset(ef_inst)